In [ ]:
#| default_exp core

# FastCDP API
> Source and API details

In [ ]:
#| export
from fastcore.utils import *
import websockets, json, platform, asyncio, inspect, base64

In [ ]:
__file__ = (Path.cwd().parent)/'fastcdp'/'core.py'
__file__

Path('/Users/jhoward/aai-ws/fastcdp/fastcdp/core.py')

In [ ]:
#| export
_path = Path(__file__).parent

_bp = (_path/'browser_protocol.json').read_json()
_jp = (_path/'js_protocol.json').read_json()

_cdp_domains = _bp['domains'] + _jp['domains']

In [ ]:
len(_cdp_domains), [d['domain'] for d in _cdp_domains[:5]]

(55, ['Accessibility', 'Animation', 'Audits', 'Autofill', 'BackgroundService'])

In [ ]:
#| export
def cdp_search(q:str):
    "Search CDP domains and commands by name or description"
    q = q.lower()
    res = []
    for d in _cdp_domains:
        for cmd in d.get('commands', []):
            desc = cmd.get('description', '')
            if q in cmd['name'].lower() or q in desc.lower():
                res.append(f"{d['domain']}.{cmd['name']}: {desc[:120]}")
        for evt in d.get('events', []):
            desc = evt.get('description', '')
            if q in evt['name'].lower() or q in desc.lower():
                res.append(f"  evt {d['domain']}.{evt['name']}: {desc[:120]}")
    return '\n'.join(res)

In [ ]:
cdp_search('target')[:100]

'Audits.checkFormsIssues: Runs the form issues check for the target page. Found issues are reported\nu'

In [ ]:
#| export
def _lower1(s): return s[0].lower() + s[1:]
def _upper1(s): return s[0].upper() + s[1:]

_chrome_paths = dict(
    Darwin='Library/Application Support/Google/Chrome/DevToolsActivePort',
    Linux ='.config/google-chrome/DevToolsActivePort')

In [ ]:
#| export
class CDP:
    "Chrome DevTools Protocol connection with event support"
    def __init__(self, port=9222, debug=False):
        self._id,self._pending,self._events = 0,{},{}
        store_attr()

    @classmethod
    async def connect(cls, port=9222):
        self = cls(port=port)
        p = Path.home()/(_chrome_paths.get(platform.system()) or _chrome_paths['Linux'])
        lines = p.read_text().strip().split('\n')
        self.ws = await websockets.connect(f'ws://127.0.0.1:{lines[0]}{lines[1]}')
        self._reader = asyncio.create_task(self._read_loop())
        self._keep = asyncio.create_task(self._keepalive())
        return self

    async def _read_loop(self):
        try:
            async for raw in self.ws:
                msg = json.loads(raw)
                if 'id' in msg:
                    if (fut := self._pending.pop(msg['id'], None)): fut.set_result(msg)
                elif 'method' in msg:
                    if self.debug: print(f"EVT: {msg['method']} sid={msg.get('sessionId','')[:8]}")
                    for q in self._events.get(msg['method'], []): q.put_nowait(msg)
        except websockets.ConnectionClosed: pass

    async def _keepalive(self):
        while True:
            try: await self('Runtime.evaluate', expression='1')
            except: break
            await asyncio.sleep(30)

    async def __call__(self, method, sid=None, **params):
        self._id += 1
        msg = dict(id=self._id, method=method)
        if params: msg['params'] = params
        if sid: msg['sessionId'] = sid
        fut = asyncio.get_event_loop().create_future()
        self._pending[self._id] = fut
        await self.ws.send(json.dumps(msg))
        r = await fut
        if 'error' in r: raise RuntimeError(f"{method}: {r['error']}")
        res = r.get('result', {})
        return first(res.values()) if len(res) == 1 else res

    async def close(self):
        self._keep.cancel()
        self._reader.cancel()
        await self.ws.close()

    def __dir__(self): return super().__dir__() + [_lower1(o) for o in _domains.keys()]

    def __getattr__(self, domain):
        if domain.startswith('_'): raise AttributeError()
        return CDPDomain(self, _upper1(domain))

In [ ]:
# await cdp.close()

In [ ]:
cdp = await CDP.connect()

In [ ]:
#| export
@patch(as_prop=True)
async def pages(self:CDP):
    return [t for t in (await self('Target.getTargets')) if t['type'] == 'page']

In [ ]:
ps = await cdp.pages
pg = next(p for p in ps if 'example.com' in p['url'])
pg['title']

'4. Example Domain'

In [ ]:
#| export
_domains = {d['domain']: d for d in _cdp_domains}

def _find_cmd(domain, method):
    d = _domains.get(domain, {})
    return first(c for c in d.get('commands', []) if c['name'] == method)

def _cmd_sig(cmd):
    _P = inspect.Parameter
    ps = [_P(p['name'], _P.KEYWORD_ONLY, default=None if p.get('optional') else _P.empty) for p in cmd.get('parameters', [])]
    return inspect.Signature([_P('sid', _P.KEYWORD_ONLY, default=None)] + ps)

def _cmd_doc(domain, method, cmd):
    doc = f"{domain}.{method}"
    if cmd.get('description'): doc += f" - {cmd['description']}"
    for p in cmd.get('parameters', []):
        opt = ' (optional)' if p.get('optional') else ''
        doc += f"\n  {p['name']}{opt}: {p.get('description', '')}"
    return doc

In [ ]:
#| export
class CDPMethod:
    def __init__(self, cdp, domain, method):
        store_attr()
        if (cmd := _find_cmd(domain, method)):
            self.__doc__ = _cmd_doc(domain, method, cmd)
            self.__signature__ = _cmd_sig(cmd)

    async def __call__(self, sid=None, **kw): return await self.cdp(f'{self.domain}.{self.method}', sid=sid, **kw)

class CDPDomain:
    def __init__(self, cdp, domain): store_attr()

    def __getattr__(self, method):
        if method.startswith('_'): raise AttributeError()
        return CDPMethod(self.cdp, self.domain, method)

    def __dir__(self):
        d = _domains.get(self.domain, {})
        return [c['name'] for c in d.get('commands', [])] + [e['name'] for e in d.get('events', [])]

In [ ]:
#| export
@patch
async def attach(self:CDP, tid):
    return await self.target.attachToTarget(targetId=tid, flatten=True)

@patch
async def eval(self:CDP, expr, sid=None):
    return (await self.runtime.evaluate(sid=sid, expression=expr)).get('value')

In [ ]:
tid = pg['targetId']
sid = await cdp.attach(tid)
await cdp.eval('document.title', sid)

'4. Example Domain'

In [ ]:
#| export
@patch
def on(self:CDP, event):
    q = asyncio.Queue()
    self._events.setdefault(event, []).append(q)
    return q

@patch
async def wait_event(self:CDP, event, timeout=10):
    q = self.on(event)
    try: return await asyncio.wait_for(q.get(), timeout)
    finally: self._events[event].remove(q)

In [ ]:
#| export
@patch
async def wait_load(self:CDP, sid=None, timeout=10):
    while True:
        e = await self.wait_event('Page.lifecycleEvent', timeout=timeout)
        print(e)
        if e['params'].get('name') == 'networkIdle': return

@patch
async def wait_for(self:CDP, expr, sid=None, timeout=10):
    "Wait for JS expression to be truthy, return its value"
    deadline = asyncio.get_event_loop().time() + timeout
    while asyncio.get_event_loop().time() < deadline:
        r = await self.eval(expr, sid)
        if r: return r
        await asyncio.sleep(0.05)
    raise TimeoutError(f'Timed out waiting for: {expr}')

@patch
async def wait_for_selector(self:CDP, sel, sid=None, timeout=10):
    "Wait for CSS selector to match an element"
    return await self.wait_for(f'!!document.querySelector("{sel}")', sid, timeout)

In [ ]:
t = await cdp.target.createTarget(url='about:blank')
sid = await cdp.attach(t)
page = await cdp.page.enable(sid=sid)

await cdp.page.navigate(sid=sid, url='https://httpbin.org/forms/post')
await cdp.wait_for_selector('form', sid)

True

In [ ]:
await cdp.target.closeTarget(targetId=t)

True

In [ ]:
#| export
class PageDomain:
    def __init__(self, sid, domain): store_attr()

    def __getattr__(self, name):
        if name.startswith('_'): raise AttributeError(name)
        m = getattr(self.domain, name)
        async def _f(**kw): return await m(sid=self.sid, **kw)
        return _f

class Page:
    def __init__(self, cdp, t, sid): store_attr()

    @classmethod
    async def new(cls, cdp, t):
        sid = await cdp.attach(t)
        self = cls(cdp, t, sid)
        await cdp.page.enable(sid=sid)
        return self

    def __getattr__(self, name):
        if name.startswith('_'): raise AttributeError(name)
        o = getattr(self.cdp, name)
        if isinstance(o, CDPDomain): return PageDomain(self.sid, o)
        if not callable(o): return o
        async def _f(*a, **kw): return await o(*a, sid=self.sid, **kw)
        return _f

    async def close(self): return await self.cdp.target.closeTarget(targetId=self.t)

In [ ]:
#| export
@patch
async def new_page(self:CDP):
    "Create a new tab, return Page"
    t = await self.target.createTarget(url='about:blank')
    return await Page.new(self, t)

In [ ]:
page = await cdp.new_page()
await page.page.navigate(url='https://httpbin.org/forms/post')
await page.wait_for_selector('form')

True

In [ ]:
await page.close()

True

In [ ]:
#| export
def cdp_yolo():
    "Allow all CDP classes in safepyrun"
    allow({CDP:..., CDPDomain:..., CDPMethod:...})
    allow({PageDomain:..., Page:...})

In [ ]:
#| export
class Event:
    "Context manager for CDP event subscription"
    def __init__(self, cdp, name): store_attr()
    async def __aenter__(self):
        self.q = self.cdp.on(self.name)
        return self.q
    async def __aexit__(self, *args): self.cdp._events[self.name].remove(self.q)

@patch
def event(self:CDP, name): return Event(self, name)

In [ ]:
#| export
@patch
async def goto(self:CDP, url, sid=None, timeout=10):
    async with self.event('Page.loadEventFired') as q:
        await self.page.navigate(sid=sid, url=url)
        await asyncio.wait_for(q.get(), timeout=timeout)
    await self.wait_for_selector('body', sid)

In [ ]:
page = await cdp.new_page()
await page.goto('https://httpbin.org/forms/post')
await page.wait_for('document.title')

'6. httpbin.org/forms/post'

In [ ]:
#| export
@patch
async def screenshot(self:CDP, sid=None):
    from IPython.display import Image
    b64 = await self.page.captureScreenshot(sid=sid, format='png')
    return Image(base64.b64decode(b64))

In [ ]:
img = await page.screenshot()
# img

In [ ]:
await page.accessibility.enable()
tree = await page.accessibility.getFullAXTree()
len(tree)

90

In [ ]:
tree[0]

{'nodeId': '2',
 'ignored': False,
 'role': {'type': 'internalRole', 'value': 'RootWebArea'},
 'chromeRole': {'type': 'internalRole', 'value': 144},
 'name': {'type': 'computedString',
  'value': '6. httpbin.org/forms/post',
  'sources': [{'type': 'relatedElement', 'attribute': 'aria-labelledby'},
   {'type': 'attribute', 'attribute': 'aria-label'},
   {'type': 'attribute', 'attribute': 'aria-label', 'superseded': True},
   {'type': 'relatedElement',
    'value': {'type': 'computedString', 'value': '6. httpbin.org/forms/post'},
    'nativeSource': 'title'}]},
 'properties': [{'name': 'focusable',
   'value': {'type': 'booleanOrUndefined', 'value': True}},
  {'name': 'url',
   'value': {'type': 'string', 'value': 'https://httpbin.org/forms/post'}}],
 'childIds': ['19'],
 'backendDOMNodeId': 2,
 'frameId': 'DED6AD4E409638E475109A9463285F29'}

In [ ]:
await page.close()

True

In [ ]:
await cdp.close()

Approach to having LLM interact with screen:

1. `Accessibility.enable` on the session
2. `Accessibility.getFullAXTree` to get all nodes
3. Filter to interactive/visible nodes, assign ref numbers
4. Format as that numbered list
5. Give the LLM that list + optionally a screenshot